# General-instruction eval — base Instruct vs your SFT model

Checks whether SFT **preserved general ability** (the catastrophic-forgetting / co-training
check). Loads **`allenai/OLMo-2-0425-1B-Instruct`** and your **SFT LoRA adapter**, generates a
response from **each** model to a bank of general (non-pedagogy) instructions with **no system
prompt**, and saves everything to `general_eval_results.jsonl`.

**Scoring is done later, offline** (blind pairwise base-vs-SFT), so **no API key is needed here.**

Run-all: set the GPU runtime, set `SFT_MODEL` (defaults to the Drive path below), Run All.
The only interactive step is the Google Drive auth popup.

In [ ]:
# 1. Install
!pip -q install -U transformers accelerate peft safetensors
!pip -q uninstall -y torchao   # peft rejects Colab's old torchao 0.10; we don't use it -> remove it
import torch, transformers
print("transformers", transformers.__version__, "| cuda:", torch.cuda.is_available())

In [ ]:
# 2. Config
BASE_MODEL = "allenai/OLMo-2-0425-1B-Instruct"

# Your SFT LoRA adapter. Point at the CHECKPOINT folder that holds adapter_config.json.
# From your Drive save that is:
SFT_MODEL = "/content/drive/MyDrive/olmo2_socratic_sft/instruct/olmo2-1b-socratic-tutor-instruct/checkpoint-923"
# If the adapter isn't on Drive, upload the folder and set the local path, or use a HF repo id.

MOUNT_DRIVE = True      # needed to read the adapter from Drive (and to save results back)
NO_SYSTEM_PROMPT = True # general default behavior: send NO system message
GEN_MAX_NEW = 400
GEN_TEMP, GEN_TOP_P = 0.7, 0.9
BATCH_SIZE = 16
SEED = 0
RESULTS_PATH = "general_eval_results.jsonl"

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

In [ ]:
# 3. Prompt bank (general, non-pedagogy) — embedded so no upload needed
PROMPTS = [{"id": "fact_capital", "category": "factual_qa", "prompt": "What is the capital of Australia?"},
           {"id": "fact_author", "category": "factual_qa", "prompt": "Who wrote the novel \"Pride and Prejudice\"?"},
           {"id": "fact_boiling", "category": "factual_qa", "prompt": "What is the boiling point of water at sea level in degrees Celsius?"},
           {"id": "math_mult", "category": "math_solve", "prompt": "What is 17 * 23? Reply with just the number."},
           {"id": "math_discount", "category": "math_solve", "prompt": "A shirt costs $40 and is 25% off. What is the final price?"},
           {"id": "math_linear", "category": "math_solve", "prompt": "Solve for x: 3x + 6 = 21."},
           {"id": "code_fib", "category": "code", "prompt": "Write a Python function that returns the nth Fibonacci number."},
           {"id": "code_reverse", "category": "code", "prompt": "Write a one-line Python expression to reverse the string s."},
           {"id": "code_sql", "category": "code", "prompt": "Write a SQL query to select all rows from a table called users where age is greater than 30."},
           {"id": "sum_photosynthesis", "category": "summarization", "prompt": "Summarize this in one sentence: Photosynthesis is the process by which green plants use sunlight to synthesize food from carbon dioxide and water, releasing oxygen as a byproduct."},
           {"id": "sum_romeo", "category": "summarization", "prompt": "Summarize the plot of Romeo and Juliet in two sentences."},
           {"id": "sum_watercycle", "category": "summarization", "prompt": "Give a TL;DR in one sentence: Water evaporates from oceans and lakes, condenses into clouds, falls as precipitation, and flows back to the sea, repeating continuously."},
           {"id": "reason_syllogism", "category": "reasoning", "prompt": "If all Bloops are Razzies and all Razzies are Lazzies, are all Bloops Lazzies? Explain briefly."},
           {"id": "reason_batball", "category": "reasoning", "prompt": "A bat and a ball cost $1.10 in total. The bat costs $1.00 more than the ball. How much does the ball cost?"},
           {"id": "reason_order", "category": "reasoning", "prompt": "Tom is taller than Sam. Sam is taller than Alex. Who is the shortest?"},
           {"id": "creative_poem", "category": "creative", "prompt": "Write a two-line poem about the ocean."},
           {"id": "creative_tweet", "category": "creative", "prompt": "Write a short, upbeat tweet announcing the opening of a new coffee shop."},
           {"id": "creative_names", "category": "creative", "prompt": "Give me three creative names for a bakery."},
           {"id": "edit_formal", "category": "rewriting", "prompt": "Rewrite this sentence to be more formal: \"hey can u send me that file asap\""},
           {"id": "edit_grammar", "category": "rewriting", "prompt": "Fix the grammar in this sentence: \"She don't like when it rains on the weekends.\""},
           {"id": "edit_concise", "category": "rewriting", "prompt": "Rewrite this to be more concise: \"Due to the fact that it was raining outside, we made the decision to stay inside the house.\""},
           {"id": "extract_contact", "category": "extraction", "prompt": "Extract the email address and phone number from this text: \"Reach me at jane@acme.io or 555-2020.\""},
           {"id": "extract_sentiment", "category": "extraction", "prompt": "Is this review positive or negative? \"The food was cold and the service was slow.\""},
           {"id": "extract_list", "category": "extraction", "prompt": "List only the fruits mentioned: \"I bought apples, a chair, bananas, and grapes.\""},
           {"id": "howto_omelette", "category": "howto", "prompt": "How do I make a basic omelette? List the steps."},
           {"id": "howto_sleep", "category": "howto", "prompt": "Give me three tips for sleeping better."},
           {"id": "howto_affect", "category": "howto", "prompt": "What is a simple way to remember the difference between \"affect\" and \"effect\"?"},
           {"id": "explain_prime", "category": "explanation", "prompt": "In one sentence, what is a prime number?"},
           {"id": "explain_api", "category": "explanation", "prompt": "Explain what an API is to a non-technical person in two sentences."},
           {"id": "explain_sky", "category": "explanation", "prompt": "Briefly, why is the sky blue?"},
           {"id": "constraint_capitals", "category": "constraint", "prompt": "List exactly 3 European capital cities, comma-separated, with no other text."},
           {"id": "constraint_yesno", "category": "constraint", "prompt": "Respond with only the word \"yes\" or \"no\": Is 7 a prime number?"},
           {"id": "constraint_noe", "category": "constraint", "prompt": "Write a sentence about dogs that does not contain the letter \"e\"."},
           {"id": "plan_pasta", "category": "planning", "prompt": "Make a simple 3-item grocery list for cooking pasta with tomato sauce."},
           {"id": "plan_convert", "category": "planning", "prompt": "Convert 2.5 hours into minutes and explain the calculation in one line."},
           {"id": "plan_budget", "category": "planning", "prompt": "I have $50 and want to buy notebooks that cost $4 each. How many can I buy, and how much money is left?"}]
print(len(PROMPTS), 'prompts |', sorted(set(p['category'] for p in PROMPTS)))

In [ ]:
# 4. Model loader (LoRA adapter auto-detected + merged onto its base)
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed
from peft import PeftConfig, PeftModel
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8) else torch.float16

def _is_adapter(mid):
    try:
        PeftConfig.from_pretrained(mid); return True
    except Exception:
        return False

def load_model(model_id, base_fallback=BASE_MODEL):
    is_ad = _is_adapter(model_id)
    base_id = base_fallback
    if is_ad:
        base_id = PeftConfig.from_pretrained(model_id).base_model_name_or_path or base_fallback
    print(f"loading {model_id}" + (f"  (LoRA on {base_id})" if is_ad else ""))
    try:
        tok = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    except Exception:
        tok = AutoTokenizer.from_pretrained(base_id, trust_remote_code=True)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = "left"
    m = AutoModelForCausalLM.from_pretrained(base_id if is_ad else model_id, dtype=DTYPE, trust_remote_code=True)
    if is_ad:
        m = PeftModel.from_pretrained(m, model_id).merge_and_unload()
    return m.to(DEVICE).eval(), tok

@torch.no_grad()
def generate_all(model, tok, prompts):
    set_seed(SEED)
    msgs = []
    for p in prompts:
        conv = [] if NO_SYSTEM_PROMPT else [{"role": "system", "content": "You are a helpful assistant."}]
        conv.append({"role": "user", "content": p["prompt"]})
        msgs.append(tok.apply_chat_template(conv, tokenize=False, add_generation_prompt=True))
    out = []
    for b in range(0, len(msgs), BATCH_SIZE):
        enc = tok(msgs[b:b+BATCH_SIZE], return_tensors="pt", padding=True, truncation=True,
                  max_length=2048, add_special_tokens=False).to(DEVICE)
        gen = model.generate(**enc, do_sample=True, temperature=GEN_TEMP, top_p=GEN_TOP_P,
                             max_new_tokens=GEN_MAX_NEW, pad_token_id=tok.pad_token_id)
        new = gen[:, enc["input_ids"].shape[1]:]
        out.extend(t.strip() for t in tok.batch_decode(new, skip_special_tokens=True))
    return out

In [ ]:
# 5. Generate for BOTH models
import gc
results = [{"id": p["id"], "category": p["category"], "prompt": p["prompt"], "outputs": {}} for p in PROMPTS]

for label, mid in [("base", BASE_MODEL), ("sft", SFT_MODEL)]:
    model, tok = load_model(mid)
    print(f"generating {label} ...")
    outs = generate_all(model, tok, PROMPTS)
    for r, o in zip(results, outs):
        r["outputs"][label] = o
    del model; gc.collect(); torch.cuda.empty_cache()

print("done. sample:")
print(results[0]["prompt"])
print("  base:", results[0]["outputs"]["base"][:160])
print("  sft :", results[0]["outputs"]["sft"][:160])

In [ ]:
# 6. Save results (+ Drive backup)
import json, os, shutil
with open(RESULTS_PATH, "w", encoding="utf-8") as f:
    for r in results:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
print("wrote", RESULTS_PATH, "with", len(results), "records")

if MOUNT_DRIVE:
    dst = "/content/drive/MyDrive/olmo2_socratic_sft/instruct"
    os.makedirs(dst, exist_ok=True)
    shutil.copy(RESULTS_PATH, dst)
    print("backed up ->", os.path.join(dst, RESULTS_PATH))

# Download general_eval_results.jsonl (left file panel) and send it back for offline scoring.